# 07. 記憶體與持久化

學習如何在 LangGraph 中保存和恢復對話狀態。

---

## 🎯 學習目標

完成本章節後，您將能夠：
- ✅ 理解 Checkpointer 的作用
- ✅ 使用 MemorySaver 實現記憶體存儲
- ✅ 實現多輪對話的狀態保持
- ✅ 使用 thread_id 管理多個會話

---

## 📊 記憶體機制架構

```
┌─────────────────────────────────────────────────────────┐
│                    記憶體機制架構                        │
├─────────────────────────────────────────────────────────┤
│                                                         │
│   ┌────────────────┐        ┌────────────────────────┐ │
│   │   用戶請求 1   │ ─────▶ │                        │ │
│   └────────────────┘        │                        │ │
│                             │      LangGraph         │ │
│   ┌────────────────┐        │        +               │ │
│   │   用戶請求 2   │ ─────▶ │    Checkpointer       │ │
│   └────────────────┘        │                        │ │
│                             │                        │ │
│   ┌────────────────┐        └───────────┬────────────┘ │
│   │   用戶請求 3   │ ─────▶             │              │
│   └────────────────┘                    │              │
│                                         ▼              │
│                             ┌────────────────────────┐ │
│                             │    狀態儲存 (State)     │ │
│                             │  • 對話歷史            │ │
│                             │  • 上下文資料          │ │
│                             │  • 執行狀態            │ │
│                             └────────────────────────┘ │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

### Checkpointer 類型

| 類型 | 用途 | 持久性 |
|------|------|--------|
| `MemorySaver` | 開發測試 | ❌ 程式結束就消失 |
| `SqliteSaver` | 本地持久化 | ✅ 寫入檔案 |
| `PostgresSaver` | 生產環境 | ✅ 寫入資料庫 |

In [1]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

---

## 7.1 建立帶記憶的聊天機器人

### 關鍵概念

| 概念 | 說明 |
|------|------|
| `thread_id` | 會話識別碼，區分不同對話 |
| `config` | 傳遞給 invoke 的配置 |
| `checkpointer` | 狀態存儲器 |

In [2]:
class ChatState(TypedDict):
    """對話狀態"""
    messages: Annotated[list, add_messages]  # 對話歷史自動累加

def chatbot(state: ChatState) -> dict:
    """模擬聊天機器人"""
    messages = state["messages"]
    
    # 取得最後一條訊息
    if messages:
        last = messages[-1]
        content = last.content if hasattr(last, 'content') else str(last)
    else:
        content = "Hello"
    
    # 顯示對話歷史長度
    print(f"  📝 當前對話歷史: {len(messages)} 條訊息")
    
    # 生成回覆
    response = f"收到第 {len(messages)} 條訊息: '{content[:30]}...'"
    
    return {"messages": [{"role": "assistant", "content": response}]}

print("✅ 聊天機器人定義完成")

✅ 聊天機器人定義完成


In [3]:
# 建構圖
graph = StateGraph(ChatState)
graph.add_node("chatbot", chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

# 🔑 關鍵：添加 checkpointer
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

print("✅ 帶記憶的圖編譯完成")
print("💡 使用 MemorySaver 作為狀態存儲")

✅ 帶記憶的圖編譯完成
💡 使用 MemorySaver 作為狀態存儲


---

## 7.2 多輪對話示範

In [4]:
# 定義 thread_id 來識別對話
config = {"configurable": {"thread_id": "conversation-1"}}

print("💬 多輪對話示範（使用同一個 thread_id）：")
print("=" * 50)

# 第一輪
print("\n--- 第 1 輪 ---")
result = app.invoke(
    {"messages": [{"role": "user", "content": "你好！我叫小明"}]},
    config
)
print(f"回覆: {result['messages'][-1]}")

# 第二輪（記憶會保留）
print("\n--- 第 2 輪 ---")
result = app.invoke(
    {"messages": [{"role": "user", "content": "你還記得我的名字嗎？"}]},
    config
)
print(f"回覆: {result['messages'][-1]}")

# 第三輪
print("\n--- 第 3 輪 ---")
result = app.invoke(
    {"messages": [{"role": "user", "content": "幫我總結我們的對話"}]},
    config
)
print(f"回覆: {result['messages'][-1]}")

print("\n" + "=" * 50)
print(f"📊 總對話歷史: {len(result['messages'])} 條訊息")

💬 多輪對話示範（使用同一個 thread_id）：

--- 第 1 輪 ---
  📝 當前對話歷史: 1 條訊息
回覆: content="收到第 1 條訊息: '你好！我叫小明...'" additional_kwargs={} response_metadata={} id='00c702a3-ead6-4007-9479-1755e6b3efc4'

--- 第 2 輪 ---
  📝 當前對話歷史: 3 條訊息
回覆: content="收到第 3 條訊息: '你還記得我的名字嗎？...'" additional_kwargs={} response_metadata={} id='14272021-e43d-4265-81ff-27b9d6810f0b'

--- 第 3 輪 ---
  📝 當前對話歷史: 5 條訊息
回覆: content="收到第 5 條訊息: '幫我總結我們的對話...'" additional_kwargs={} response_metadata={} id='f3d031bf-ec47-4d2b-969e-2a18485ee1f9'

📊 總對話歷史: 6 條訊息


---

## 7.3 多會話管理

### 不同 thread_id = 不同對話

```
thread-1: 用戶 A 的對話 ──▶ 獨立的記憶
thread-2: 用戶 B 的對話 ──▶ 獨立的記憶
thread-3: 用戶 C 的對話 ──▶ 獨立的記憶
```

In [5]:
print("💬 多會話管理示範：")
print("=" * 50)

# 用戶 A 的對話
config_a = {"configurable": {"thread_id": "user-A"}}
print("\n👤 用戶 A:")
result_a = app.invoke(
    {"messages": [{"role": "user", "content": "我是用戶 A"}]},
    config_a
)

# 用戶 B 的對話（完全獨立）
config_b = {"configurable": {"thread_id": "user-B"}}
print("\n👤 用戶 B:")
result_b = app.invoke(
    {"messages": [{"role": "user", "content": "我是用戶 B"}]},
    config_b
)

# 用戶 A 繼續對話（記憶延續）
print("\n👤 用戶 A 繼續:")
result_a2 = app.invoke(
    {"messages": [{"role": "user", "content": "這是我的第二條訊息"}]},
    config_a
)

print("\n" + "=" * 50)
print(f"用戶 A 對話記錄: {len(result_a2['messages'])} 條")
print(f"用戶 B 對話記錄: {len(result_b['messages'])} 條")

💬 多會話管理示範：

👤 用戶 A:
  📝 當前對話歷史: 1 條訊息

👤 用戶 B:
  📝 當前對話歷史: 1 條訊息

👤 用戶 A 繼續:
  📝 當前對話歷史: 3 條訊息

用戶 A 對話記錄: 4 條
用戶 B 對話記錄: 2 條


---

## 7.4 獲取歷史狀態

In [6]:
print("📜 獲取歷史狀態：")
print("=" * 50)

# 獲取特定 thread 的當前狀態
config_check = {"configurable": {"thread_id": "conversation-1"}}

try:
    state_snapshot = app.get_state(config_check)
    print(f"\n📋 Thread 'conversation-1' 狀態:")
    print(f"   訊息數: {len(state_snapshot.values.get('messages', []))}")
    
    # 顯示訊息摘要
    for i, msg in enumerate(state_snapshot.values.get('messages', [])[:5]):
        content = msg.content if hasattr(msg, 'content') else str(msg)[:30]
        print(f"   [{i+1}] {content[:40]}...")
except Exception as e:
    print(f"   無法獲取狀態: {e}")

📜 獲取歷史狀態：

📋 Thread 'conversation-1' 狀態:
   訊息數: 6
   [1] 你好！我叫小明...
   [2] 收到第 1 條訊息: '你好！我叫小明...'...
   [3] 你還記得我的名字嗎？...
   [4] 收到第 3 條訊息: '你還記得我的名字嗎？...'...
   [5] 幫我總結我們的對話...


---

## 7.5 進階：帶計數器的記憶狀態

In [7]:
class AdvancedState(TypedDict):
    """進階狀態：包含多個追蹤欄位"""
    messages: Annotated[list, add_messages]
    interaction_count: int       # 互動次數
    last_topic: str              # 最後討論的話題

def advanced_bot(state: AdvancedState) -> dict:
    count = state.get("interaction_count", 0) + 1
    last = state["messages"][-1] if state["messages"] else None
    content = last.content if last and hasattr(last, 'content') else "開始"
    
    # 簡單的話題偵測
    if "天氣" in content:
        topic = "天氣"
    elif "程式" in content or "code" in content.lower():
        topic = "程式設計"
    else:
        topic = "一般對話"
    
    response = f"[第{count}次互動] 話題: {topic}"
    print(f"  📊 互動 #{count}, 話題: {topic}")
    
    return {
        "messages": [{"role": "assistant", "content": response}],
        "interaction_count": count,
        "last_topic": topic
    }

# 建構並編譯
adv_graph = StateGraph(AdvancedState)
adv_graph.add_node("bot", advanced_bot)
adv_graph.add_edge(START, "bot")
adv_graph.add_edge("bot", END)

adv_memory = MemorySaver()
adv_app = adv_graph.compile(checkpointer=adv_memory)

print("✅ 進階記憶機器人就緒")

✅ 進階記憶機器人就緒


In [8]:
config = {"configurable": {"thread_id": "advanced-1"}}

print("📊 進階記憶測試：")
print("=" * 50)

messages = [
    "今天天氣如何？",
    "教我寫程式",
    "謝謝你的幫助"
]

for msg in messages:
    result = adv_app.invoke(
        {"messages": [{"role": "user", "content": msg}], "interaction_count": 0, "last_topic": ""},
        config
    )

print("\n" + "=" * 50)
print(f"總互動次數: {result['interaction_count']}")
print(f"最後話題: {result['last_topic']}")

📊 進階記憶測試：
  📊 互動 #1, 話題: 天氣
  📊 互動 #1, 話題: 程式設計
  📊 互動 #1, 話題: 一般對話

總互動次數: 1
最後話題: 一般對話


---

## 💡 重點回顧

### Checkpointer 使用方式

```python
# 1. 建立 checkpointer
memory = MemorySaver()

# 2. 編譯時傳入
app = graph.compile(checkpointer=memory)

# 3. 執行時指定 thread_id
config = {"configurable": {"thread_id": "my-thread"}}
result = app.invoke(input, config)
```

### 狀態管理 API

| API | 用途 |
|-----|------|
| `app.get_state(config)` | 獲取當前狀態 |
| `app.get_state_history(config)` | 獲取歷史狀態 |
| `app.update_state(config, values)` | 更新狀態 |

### 使用場景

- 💬 多輪對話保持上下文
- 👥 多用戶會話隔離
- 📊 追蹤用戶行為統計
- 🔄 支持對話恢復

---

## 📝 練習題

1. **對話摘要**：當對話超過 10 條時，自動生成摘要
2. **用戶偏好**：記住用戶的語言偏好（中文/英文）
3. **會話過期**：實作 30 分鐘無活動自動清除記憶
4. **多 Agent 記憶**：讓多個 Agent 共享同一個記憶

---

下一步：[08. 人機協作](08_human_in_loop.ipynb)